# [대학원 머신러닝 프로젝트] AI-Hub 국내 여행로그 데이터 기반 여행 소비 지출액 예측
---
본 노트북은 과제 평가 기준표(1~5단계)에 맞춰 작성된 전체 분석 파이프라인 실습 코드입니다.

1. **데이터셋 획득 및 문제 정의**: 입력변수(X 13개)와 출력변수(Y 총 지출액) 정의
2. **데이터 분할 및 비교**: Train/Test (8:2) vs Train/Val/Test (6:2:2) 비교 분석
3. **하이퍼파라미터 조정**: Grid Search 기반 최적 모델 튜닝
4. **데이터 스케일링 및 Data Leakage 방지**: Raw vs Standard vs Min-Max vs Robust 비교
5. **최종 성능 평가**: Test 세트 1회 최종 검증 (MAE, MSE, RMSE, R²)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))
from src.data.travel_dataset import load_travel_dataset
from src.models.ml_pipeline import TravelExpenditurePipeline

# Visual styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터셋 획득 및 문제 정의 (단계 1)

In [ ]:
df = load_travel_dataset()
print(f'총 데이터 수: {df.shape[0]}행, {df.shape[1]}열')
display(df.head())
display(df['TOTAL_EXPENDITURE'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['TOTAL_EXPENDITURE'], bins=40, kde=True, ax=axes[0], color='royalblue')
axes[0].set_title('총 지출액 분포 (우측 왜도 및 이상치)', fontsize=14)
axes[0].set_xlabel('총 소비 지출액 (원)')

sns.boxplot(x='TRAVEL_DAYS', y='TOTAL_EXPENDITURE', data=df, ax=axes[1], palette='Set2')
axes[1].set_title('여행 기간(박수)별 지출액 분포', fontsize=14)
axes[1].set_xlabel('여행 기간 (일)')
axes[1].set_ylabel('총 소비 지출액 (원)')
plt.tight_layout()
plt.show()

## 2. 5단계 파이프라인 일괄 실행 (단계 2 ~ 5)

In [ ]:
pipeline = TravelExpenditurePipeline(df=df, random_state=42)

# Step 2: 2분할 vs 3분할 비교
split_results = pipeline.execute_split_comparison()

# Step 4: 스케일러 3종 비교 및 Leakage 방지
scaling_results = pipeline.execute_scaling_comparison()

# Step 3: 하이퍼파라미터 튜닝 (Grid Search)
tuning_results = pipeline.execute_hyperparameter_tuning()

# Step 5: 최종 1회 Test 평가
final_metrics = pipeline.execute_final_evaluation()

## 3. 스케일러별 성능 비교 시각화 (단계 4)

In [ ]:
scale_metrics = pipeline.results['scaling_comparison']['metrics']
names = list(scale_metrics.keys())
rmses = [scale_metrics[n]['Val_RMSE'] for n in names]
r2s = [scale_metrics[n]['Val_R2'] for n in names]

fig, ax1 = plt.subplots(figsize=(10, 5))
x = np.arange(len(names))
width = 0.35

ax1.bar(x - width/2, rmses, width, label='Val RMSE (원)', color='coral')
ax1.set_ylabel('RMSE (원)', color='coral', fontsize=12)
ax1.set_xticks(x)
ax1.set_xticklabels(names, fontsize=11)

ax2 = ax1.twinx()
ax2.plot(x + width/2, r2s, color='navy', marker='o', linewidth=2.5, label='Val R²')
ax2.set_ylabel('R² Score', color='navy', fontsize=12)

plt.title('스케일러별 검증 데이터셋 성능 비교 (RobustScaler 최고 성능)', fontsize=14)
plt.tight_layout()
plt.show()